<a href="https://colab.research.google.com/github/FridaOyucho/HTS-Model-/blob/main/HTSModelDataPrep19082026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# This script begins with tables exported from the National Data warehouse and
#concludes with the creation of ML-ready datasets that are  then read in by
# model training scripts. By ML-ready datasets, we mean datasets in which each
# row is an observation with a labeled outcome.

# The sript proceeds through three steps:
# 1) Combining and filtering original tables
# 2) Data Cleaning and feature generation
# 3) Missing data imputation

# First, load necessary packages
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
from statsmodels.imputation.mice import MICEData


                       **  Combine and filter original tables**

In [2]:
# This script assumes user is accessing tables extracted and loaded from ODS Database.

#Load tables
eligibility = pd.read_csv('eligibility.csv', low_memory=False)
tests = pd.read_csv('tests.csv', low_memory=False)
xwalk = pd.read_csv('ActiveEMRSites_07152026.csv', low_memory=False)
clients = pd.read_csv('clients.csv', low_memory=False)

In [3]:
# Preview content of tables
rows,cols = eligibility.shape
print(f"There are {rows} rows and {cols} columns in the eligibility table")

rows,cols = tests.shape
print(f"There are {rows} rows and {cols} columns in the tests table")

rows,cols = xwalk.shape
print(f"There are {rows} rows and {cols} columns in the xwalk table")

rows,cols = clients.shape
print(f"There are {rows} rows and {cols} columns in the clients table")

There are 2709027 rows and 87 columns in the eligibility table
There are 3157391 rows and 34 columns in the tests table
There are 2456 rows and 19 columns in the xwalk table
There are 12199267 rows and 6 columns in the clients table


In [4]:

# Tests
# Filter tests to results that are positive or negative (exclude inconclusive)
display(tests.columns)

tests = tests[tests['FinalTestResult'] != 'Inconclusive']

# Select columns to keep and remove duplicates(We are keeping all columns)
cols_to_keep = ['SiteCode', 'PatientPk', 'Emr', 'Project',
      'TestDate', 'EverTestedForHiv', 'MonthsSinceLastTest',
       'ClientTestedAs', 'EntryPoint', 'TestStrategy', 'TestResult1',
       'TestResult2', 'TestResult3', 'FinalTestResult', 'PatientGivenResult',
       'TbScreening', 'ClientSelfTested', 'CoupleDiscordant', 'TestType',
       'Consent', 'Setting', 'Approach','PatientPKHash', 'ReferredForServices',
       'ReferredServices', 'LoadDate', 'PriorityPopulationType']
tests = tests[cols_to_keep]
tests = tests.drop_duplicates(subset=["SiteCode", "PatientPk"])

rows,cols = tests.shape
print(f"There are {rows} rows and {cols} columns in the tests table")


There are 2713549 rows and 27 columns in the tests table


In [6]:
#Clients
# Select columns to keep and remove duplicates. We are keeping all columns in clients dataset
display(clients.columns)

clients = clients.drop_duplicates(subset= ["SiteCode","PatientPk"])

rows,cols = clients.shape
print(f"There are {rows} rows and {cols} columns in the clients table")

There are 12199267 rows and 6 columns in the clients table


In [ ]:
# Eligibility
# Convert visitdate from character to date and filter to between April 2025 and June 2026
print(eligibility["VisitDate"].dtype)
eligibility['VisitDate'] = pd.to_datetime(eligibility['VisitDate'].astype(str).str[:10])

eligibility = eligibility[eligibility['VisitDate'] >= '2025-04-01']
eligibility = eligibility[eligibility['VisitDate'] <= '2026-06-30']

rows,cols = eligibility.shape
print(f"There are {rows} rows and {cols} columns in the eligibility table")

object
There are 2509280 rows and 87 columns in the eligibility table


In [10]:
# Select columns to keep and remove duplicates(Calculation of missingness and variance done separately)
cols = eligibility.columns
#print(cols)
cols_to_keep = ['SiteCode', 'PatientPk', 'HtsNumber', 'Emr', 'Project',
       'VisitID', 'VisitDate', 'PopulationType', 'KeyPopulation',
       'PriorityPopulation', 'Department', 'PatientType', 'IsHealthWorker',
       'RelationshipWithContact', 'TestedHIVBefore', 'WhoPerformedTest',
       'ResultOfHIV', 'DateTestedSelf', 'StartedOnART', 'EverHadSex',
       'SexuallyActive', 'NewPartner', 'PartnerHIVStatus', 'CoupleDiscordant',
       'MultiplePartners', 'NumberOfPartners', 'AlcoholSex', 'MoneySex',
       'CondomBurst', 'UnknownStatusPartner', 'KnownStatusPartner', 'Pregnant',
       'BreastfeedingMother', 'ExperiencedViolenceScreening',
       'ContactWithTBCase', 'Lethargy', 'CurrentlyOnPrep', 'CurrentlyOnPep',
       'CurrentlyHasSTI', 'SharedNeedle', 'NeedleStickInjuries',
       'TraditionalProcedures', 'ChildReasonsForIneligibility',
       'EligibleForTest', 'ReasonsForIneligibility',
       'SpecificReasonForIneligibility', 'Cough', 'DateTestedProvider',
       'Fever', 'MothersStatus', 'NightSweats', 'ReferredForTesting',
       'ResultOfHIVSelf', 'ScreenedTB', 'TBStatus', 'WeightLoss',
       'AssessmentOutcome', 'ForcedSex', 'ReceivedServices',
       'ViolenceScreeningType', 'PatientPKHash', 'HtsNumberHash', 'ID',
       'Disability', 'DisabilityType', 'HTSStrategy', 'HTSEntryPoint',
       'ReasonRefferredForTesting', 'ReasonNotReffered',
       'LoadDate']
eligibility = eligibility[cols_to_keep]
eligibility = eligibility.drop_duplicates(subset=["SiteCode", "PatientPk"])

rows,cols = eligibility.shape
print(f"There are {rows} rows and {cols} columns in the eligibility table")

There are 2340216 rows and 70 columns in the eligibility table


In [11]:
print(eligibility.columns)

Index(['SiteCode', 'PatientPk', 'HtsNumber', 'Emr', 'Project', 'VisitID',
       'VisitDate', 'PopulationType', 'KeyPopulation', 'PriorityPopulation',
       'Department', 'PatientType', 'IsHealthWorker',
       'RelationshipWithContact', 'TestedHIVBefore', 'WhoPerformedTest',
       'ResultOfHIV', 'DateTestedSelf', 'StartedOnART', 'EverHadSex',
       'SexuallyActive', 'NewPartner', 'PartnerHIVStatus', 'CoupleDiscordant',
       'MultiplePartners', 'NumberOfPartners', 'AlcoholSex', 'MoneySex',
       'CondomBurst', 'UnknownStatusPartner', 'KnownStatusPartner', 'Pregnant',
       'BreastfeedingMother', 'ExperiencedViolenceScreening',
       'ContactWithTBCase', 'Lethargy', 'CurrentlyOnPrep', 'CurrentlyOnPep',
       'CurrentlyHasSTI', 'SharedNeedle', 'NeedleStickInjuries',
       'TraditionalProcedures', 'ChildReasonsForIneligibility',
       'EligibleForTest', 'ReasonsForIneligibility',
       'SpecificReasonForIneligibility', 'Cough', 'DateTestedProvider',
       'Fever', 'MothersSta

In [13]:
# Join the three tables on SiteCode and PatientPK
merged = pd.merge(eligibility, clients, on=["SiteCode", "PatientPk"], how="left")
merged = pd.merge(merged, tests, on=["SiteCode", "PatientPk"], how="left")
merged = pd.merge(merged, xwalk, left_on=["SiteCode"], right_on=["MFL_Code"], how="left")

rows,cols = merged.shape
print(f"There are {rows} rows and {cols} columns in the merged table")

#download the merged file
import requests
import csv
merged.to_csv('merged.csv', index=False)

There are 2340216 rows and 118 columns in the merged table


In [15]:
#Rename the merged dataset to hts
hts = merged


rows,cols = hts.shape
print(f"There are {rows} rows and {cols} columns in the hts table")

There are 2340216 rows and 118 columns in the hts table


In [16]:
# check for behavioural data from EDARP Project
print(hts.groupby(['Project_x']).size().reset_index(name='Count').to_string())
hts_filtered_emr = hts[hts['Project_x'].isin(['EDARP'])]

hts_edarp = hts_filtered_emr[hts_filtered_emr['Project_x'] == 'EDARP']
print(hts_filtered_emr['Project_x'].value_counts(dropna=False).to_string())

rows,cols = hts_edarp.shape
print(f"There are {rows} rows and {cols} columns in the hts_edarp table")

#we have not dropped EDARP cases because there is data from Ecare system now supported under EDARP Project

       Project_x    Count
0         Ampath    42958
1          EDARP    40612
2  Kenya HMIS II  2256646
Project_x
EDARP    40612
There are 40612 rows and 118 columns in the hts_edarp table


In [17]:

# Now, identify missing or no-variance variables to exclude
# Get breakdown by variable
# Calculate missingness for all columns and display only those with >50% missing

high_missing_cols = []
for col_name in hts.columns:
    vals = hts[col_name]
    # Count missing values (NaN or empty strings)
    missing_count = vals.isnull().sum() + (vals == '').sum()
    total_count = len(vals)

    if total_count > 0:
        missing_percentage = round(100 * (missing_count / total_count), 2)
    else:
        missing_percentage = 0.0 # Handle empty column case

    if missing_percentage > 50:
        high_missing_cols.append(f"{col_name}: {missing_percentage}%")

if high_missing_cols:
    for col_info in high_missing_cols:
        print(col_info)
else:
    print("No columns found with more than 50% missing values.")

KeyPopulation: 92.77%
PriorityPopulation: 96.87%
RelationshipWithContact: 90.02%
DateTestedSelf: 98.91%
StartedOnART: 99.92%
CoupleDiscordant_x: 97.73%
NumberOfPartners: 88.82%
CurrentlyOnPep: 95.89%
SharedNeedle: 98.4%
NeedleStickInjuries: 96.92%
ChildReasonsForIneligibility: 99.37%
ReasonsForIneligibility: 99.86%
SpecificReasonForIneligibility: 100.0%
DateTestedProvider: 67.24%
MothersStatus: 97.35%
ReferredForTesting: 98.1%
ResultOfHIVSelf: 98.76%
AssessmentOutcome: 100.0%
ForcedSex: 99.94%
ViolenceScreeningType: 97.52%
Disability: 98.26%
DisabilityType: 99.78%
ReasonNotReffered: 94.21%
MonthsSinceLastTest: 61.71%
TestResult3: 97.67%
CoupleDiscordant_y: 94.66%
PriorityPopulationType: 95.99%
Project: 100.0%
KMPDC_reg_no: 80.0%


In [19]:
display(hts.columns)
#print(cols)

Index(['SiteCode', 'PatientPk', 'HtsNumber', 'Emr_x', 'Project_x', 'VisitID',
       'VisitDate', 'PopulationType', 'KeyPopulation', 'PriorityPopulation',
       ...
       'EMR', 'EMR_Status', 'HTS_Use', 'HTS_Deployment', 'Project', 'LoadDate',
       'InfrastructureType', 'KEPH_Level', 'KMPDC_reg_no', 'Ward'],
      dtype='object', length=118)

In [21]:
#Check for variance of categorical variables
import pandas as pd

cols = [
    'StartedOnART',
    'CurrentlyOnPrep',
    'CurrentlyOnPep',
    'SharedNeedle',
    'NeedleStickInjuries',
    'ForcedSex',
    'ContactWithTBCase',
    'Lethargy',
    'NightSweats',
    'WeightLoss',
    'Fever',
    'Cough',
    'CurrentlyHasSTI',
    'Pregnant',
    'BreastfeedingMother',
    'AlcoholSex',
    'MoneySex',
    'CondomBurst'
]

results = []

for col in cols:
    vc = hts[col].value_counts(normalize=True, dropna=False)
    dominant_pct = round(vc.iloc[0] * 100, 2)

    results.append({
        'Variable': col,
        'Unique_Values': hts[col].nunique(dropna=False),
        'Dominant_Response_%': dominant_pct
    })

variance_df = pd.DataFrame(results).sort_values(
    'Dominant_Response_%',
    ascending=False
)

print(variance_df) #will not drop because they are clinically important

               Variable  Unique_Values  Dominant_Response_%
5             ForcedSex              3                99.94
0          StartedOnART              3                99.92
8           NightSweats              3                99.08
9            WeightLoss              3                99.08
11                Cough              3                98.96
3          SharedNeedle              3                98.40
10                Fever              3                97.43
4   NeedleStickInjuries              3                96.92
7              Lethargy              3                96.44
6     ContactWithTBCase              2                96.43
2        CurrentlyOnPep              3                95.89
15           AlcoholSex              5                66.41
17          CondomBurst              6                66.31
16             MoneySex              6                65.36
1       CurrentlyOnPrep              6                50.66
13             Pregnant              6  

In [22]:
#variables to drop
variables_to_drop = ['ChildReasonsForIneligibility','ReasonsForIneligibility','SpecificReasonForIneligibility',"EverOnPrep", "EverOnPep","EverHadSTI","EverHadTB",
                     "Project_y",'AssessmentOutcome', "KMPDC_reg_no", "InfrastructureType", "HTS_Deployment", "HTS_Use", "EMR_Status","SDP_Agency"]

hts = hts.drop(variables_to_drop, errors='ignore')

rows,cols = hts.shape
print(f"There are {rows} rows and {cols} columns in the hts table")

There are 2340216 rows and 118 columns in the hts table


                                 **   Data cleaning and feature generation **

In [23]:
# Create adult/child flag for checking patterns more easily
# Create VDate as date field, DOB as date field, and Age

hts['VDate'] = pd.to_datetime(hts['VisitDate'].astype(str).str[:10])
hts['DOB'] = pd.to_datetime(hts['Dob'].astype(str).str[:10], errors='coerce')

# Calculate Age
hts['Age'] = (hts['VDate'] - hts['DOB']).dt.days // 365

# Filter out erroneous ages (Age > 0 and Age < 100)
hts = hts[hts['Age'] > 0]
hts = hts[hts['Age'] < 100]

# Create cohort column
hts['cohort'] = np.where(hts['Age'] >= 15, "Adult", "Child")

rows,cols = hts.shape
print(f"There are {rows} rows and {cols} columns in the hts table after age filtering")

There are 2339903 rows and 122 columns in the hts table after age filtering


In [24]:
## Population Type -
# If Null, then NA
hts['PopulationType'] = hts['PopulationType'].fillna('NA')

display(hts['PopulationType'].value_counts(dropna=False))

,count
PopulationType,
General Population,2072436
Key Population,174524
Priority Population,80993
NA,11860
Vulnerable Population,90


In [25]:
# Label as GP, KP, Priority

# Convert PopulationType to uppercase
hts['PopulationType'] = hts['PopulationType'].str.upper()

# Map PopulationType values to 'GP', 'KP', or 'PRIORITY'
conditions = [
    hts['PopulationType'].str.contains('GENERAL', na=False),
    hts['PopulationType'].str.contains('KEY', na=False),
    hts['PopulationType'].str.contains('PRIORITY', na=False)
]
choices = ['GP', 'KP', 'PRIORITY']
hts['PopulationType'] = np.select(conditions, choices, default=hts['PopulationType'])

In [26]:
#display(hts['KeyPopulation'].value_counts(dropna=False))

In [27]:
# We're going to use NR for not relevant throughout
# Clean labels for KPs. Create other for rare values

hts['KeyPopulation'] = hts['KeyPopulation'].astype(str).str.upper()

conditions = [
    hts['PopulationType'] != 'KP', # If PopulationType is not KP, set to NR
    (hts['PopulationType'] == 'KP') & hts['KeyPopulation'].str.contains('FEMALE', na=False),
    (hts['PopulationType'] == 'KP') & hts['KeyPopulation'].str.contains('MEN', na=False),
    (hts['PopulationType'] == 'KP') & hts['KeyPopulation'].str.contains('DRUGS', na=False),
    (hts['PopulationType'] == 'KP') & hts['KeyPopulation'].str.contains('PRISON|TRANS', na=False)
]

choices = [
    'NR',
    'FSW',
    'MSM',
    'PWID',
    'OTHER'
]

hts['KeyPopulation'] = np.select(conditions, choices, default=hts['KeyPopulation'])

#display(hts['KeyPopulation'].value_counts(dropna=False))

# Only people in KP or Priority should have a non-Null value here
display(hts[['PopulationType', 'KeyPopulation']].value_counts(dropna=False))

PopulationType         KeyPopulation
GP                     NR               2072436
KP                     FSW                98046
PRIORITY               NR                 80993
KP                     MSM                41139
                       OTHER              25334
NA                     NR                 11860
KP                     NAN                 5334
                       PWID                4671
VULNERABLE POPULATION  NR                    90
Name: count, dtype: int64

In [28]:
## Priority Population
#display(hts['PriorityPopulation'].value_counts(dropna=False))

hts['PriorityPopulation'] = hts['PriorityPopulation'].astype(str).str.upper()

conditions = [
    hts['PopulationType'] != 'PRIORITY', # If PopulationType is not PRIORITY, set to NR
    (hts['PopulationType'] == 'PRIORITY') & hts['PriorityPopulation'].str.contains('FISHER', na=False),
    (hts['PopulationType'] == 'PRIORITY') & hts['PriorityPopulation'].str.contains('ADOLESCENT ', na=False),
    (hts['PopulationType'] == 'PRIORITY') & hts['PriorityPopulation'].str.contains('MILITARY|PRISONER|TRUCK|YOUNG|FAMILIES|PEOPLE|SERVICE|WIDOWS|OTHERS', na=False)
]

choices = [
    'NR',
    'FISHERMEN',
    'AGYW',
    'OTHER'
]

hts['PriorityPopulation'] = np.select(conditions, choices, default=hts['PriorityPopulation'])

# Display the new distribution of PriorityPopulation
#display(hts['PriorityPopulation'].value_counts(dropna=False))

# Only people in KP or Priority should have a non-Null value here
display(hts[['PopulationType', 'PriorityPopulation']].value_counts(dropna=False))

PopulationType         PriorityPopulation             
GP                     NR                                 2072436
KP                     NR                                  174524
PRIORITY               AGYW                                 26398
                       OTHER                                23504
                       FISHERMEN                            23207
NA                     NR                                   11860
PRIORITY               NAN                                   7883
VULNERABLE POPULATION  NR                                      90
PRIORITY               ORPHANS AND VULNERABLE CHILDREN          1
Name: count, dtype: int64

In [29]:
# Fix population type
hts_population_conditions = [
    (hts['KeyPopulation'] == 'NR') & (hts['PriorityPopulation'] == 'NR'),
    (hts['KeyPopulation'] != 'NR'),
    (hts['PriorityPopulation'] == 'NR')
]

hts_population_choices = [
    'GP',
    'KP',
    'PRIORITY'
]

hts['PopulationType'] = np.select(
    hts_population_conditions,
    hts_population_choices,
    default=hts['PopulationType']
)

# Display the new distribution of PopulationType
display(hts['PopulationType'].value_counts(dropna=False))

,count
PopulationType,
GP,2084386
KP,174524
PRIORITY,80993


In [30]:
## Department
##display(hts['Department'].value_counts(dropna=False))

hts['Department'] = hts['Department'].astype(str).str.upper()

department_conditions = [
    hts['Department'].str.contains('IPD', na=False),
    hts['Department'].str.contains('OPD', na=False),
    hts['Department'].str.contains('PMTCT', na=False),
    hts['Department'].str.contains('VCT', na=False),
    hts['Department'].str.contains('EMERGENCY', na=False),
     hts['Department'].str.contains('COMMUNITY', na=False)
]

department_choices = [
    'IPD',
    'OPD',
    'PMTCT',
    'VCT',
    'EMERGENCY',
    'COMMUNITY'
]

hts['Department'] = np.select(department_conditions, department_choices, default='nan')
display(hts['Department'].value_counts(dropna=False))

,count
Department,
OPD,975164
PMTCT,643936
VCT,353294
nan,259437
IPD,97011
EMERGENCY,7476
COMMUNITY,3585


In [31]:
## Is Health Worker - what should 0 be? No or Null?
#display(hts['IsHealthWorker'].value_counts(dropna=False))

# If child, set to not relevant
hts['IsHealthWorker'] = hts['IsHealthWorker'].astype(str).str.upper()

conditions = [
    hts['cohort'] == 'Child',
    hts['IsHealthWorker'].str.contains('YES', na=False),
    hts['IsHealthWorker'].str.contains('NO', na=False)
]

choices = [
    'NR',
    'YES',
    'NO'
]

hts['IsHealthWorker'] = np.select(conditions, choices, default='NR')
display(hts['IsHealthWorker'].value_counts(dropna=False))

,count
IsHealthWorker,
NO,2091479
NR,154406
YES,94018


In [32]:
display(hts['RelationshipWithContact'].value_counts(dropna=False))

,count
RelationshipWithContact,
NaN,2106426
Sexual Contact,129208
Social Contact,87946
"Sexual Contact,Social Contact",8527
Needle sharing,2215
Sexual contact,1727
"Sexual Contact,Social Contact,Needle sharing",1559
Social contact,844
"Sexual Contact,None",565


In [33]:
## Relationship with Contact
#display(hts['RelationshipWithContact'].value_counts(dropna=False))

# Can be sexual, social, needle or none - create binary variables for each

hts['RelationshipWithContact'] = hts['RelationshipWithContact'].astype(str).str.upper()

# Define a helper function for creating binary columns to avoid repetition
def create_binary_flag(df, col_name, keyword):
    conditions = [
        df[col_name].str.contains(keyword, na=False),
        ~df[col_name].isin(['NULL', '', 'NAN'])
    ]
    choices = ['YES', 'NO']
    return np.select(conditions, choices, default='nan') # Changed default to 'nan' string

hts['SEXUAL'] = create_binary_flag(hts, 'RelationshipWithContact', 'SEXUAL')
hts['SOCIAL'] = create_binary_flag(hts, 'RelationshipWithContact', 'SOCIAL')
hts['NONE'] = create_binary_flag(hts, 'RelationshipWithContact', 'NONE')
hts['NEEDLE_SHARING'] = create_binary_flag(hts, 'RelationshipWithContact', 'NEEDLE')

print("SEXUAL Distribution:")
#display(hts['SEXUAL'].value_counts(dropna=False))

print("\nSOCIAL Distribution:")
#display(hts['SOCIAL'].value_counts(dropna=False))

print("\nNONE Distribution:")
#display(hts['NONE'].value_counts(dropna=False))

print("\nNEEDLE_SHARING Distribution:")
#display(hts['NEEDLE_SHARING'].value_counts(dropna=False))

SEXUAL Distribution:

SOCIAL Distribution:

NONE Distribution:

NEEDLE_SHARING Distribution:


In [34]:
## For all sexual practice variables, if under 10 years old, then classify as NR

hts['EverHadSex'] = hts['EverHadSex'].astype(str).str.upper()

conditions = [
    hts['Age'] <= 10,
    hts['EverHadSex'].str.contains('YES', na=False),
    hts['EverHadSex'].str.contains('NO', na=False)
]

choices = [
    'NR',
    'YES',
    'NO'
]

hts['EverHadSex'] = np.select(conditions, choices, default='nan') # Changed default to 'nan' string

#display(hts['EverHadSex'].value_counts(dropna=False))

In [35]:
# Sexually Active
#display(hts['SexuallyActive'].value_counts(dropna=False))


hts['SexuallyActive'] = hts['SexuallyActive'].astype(str).str.upper()
conditions = [
    hts['Age']<=10,
    hts['SexuallyActive'].str.contains('YES', na=False),
    hts['SexuallyActive'].str.contains('NO', na=False)
  ]


choices = [
            'NR',
            'YES',
            'NO'
 ]

hts['SexuallyActive']= np.select(conditions, choices, default='nan')
#display(hts['SexuallyActive'].value_counts(dropna=False))

In [36]:
# New Partner

hts['NewPartner'] = hts['NewPartner'].astype(str).str.upper()

conditions = [
    hts['Age'] <= 10,
    hts['NewPartner'].str.contains('YES', na=False),
    hts['NewPartner'].str.contains('NO', na=False)
]

choices = [
    'NR',
    'YES',
    'NO'
]

hts['NewPartner'] = np.select(conditions, choices, default='nan')
#display(hts['NewPartner'].value_counts(dropna=False))

In [37]:
#Check frequency of SexuallyActive vs New partner
cross_tab_sex_partner = pd.crosstab(hts['SexuallyActive'], hts['NewPartner'])
display(cross_tab_sex_partner)

NewPartner,NO,NR,YES,nan
SexuallyActive,,,,
NO,159383,0,12160,34792
NR,0,66202,0,0
YES,1182525,0,480497,70848
nan,7763,0,5512,320221


In [38]:
# PartnerHIVStatus
hts['PartnerHIVStatus'] = hts['PartnerHIVStatus'].astype(str).str.upper()

conditions = [
    hts['Age'] <= 10,
    hts['PartnerHIVStatus'].str.contains('POSITIVE', na=False),
    hts['PartnerHIVStatus'].str.contains('NEGATIVE', na=False),
    hts['PartnerHIVStatus'].str.contains('UNKNOWN', na=False)
]

choices = [
    'NR',
    'POSITIVE',
    'NEGATIVE',
    'UNKNOWN'
]

hts['PartnerHIVStatus'] = np.select(conditions, choices, default='nan')
display(hts['PartnerHIVStatus'].value_counts(dropna=False))

,count
PartnerHIVStatus,
UNKNOWN,1123865
NEGATIVE,673516
nan,421404
NR,66202
POSITIVE,54916


In [39]:
#Check frequency of SexuallyActive vs New PartnerHIVStatus
cross_tab_sex_partner = pd.crosstab(hts['SexuallyActive'], hts['PartnerHIVStatus'])
display(cross_tab_sex_partner)

PartnerHIVStatus,NEGATIVE,NR,POSITIVE,UNKNOWN,nan
SexuallyActive,,,,,
NO,84448,0,2880,82028,36979
NR,0,66202,0,0,0
YES,582637,0,51235,1032228,67770
nan,6431,0,801,9609,316655


In [40]:
#Quick check Multiplepartner is Yes and Sexually active is No

nrow_multiple_partners_not_sexually_active = hts[(hts['MultiplePartners'] == 'YES') & (hts['SexuallyActive'] == 'NO')].shape[0]
print(f"Number of rows where MultiplePartners is 'YES' and SexuallyActive is 'NO': {nrow_multiple_partners_not_sexually_active}")

Number of rows where MultiplePartners is 'YES' and SexuallyActive is 'NO': 9301


In [41]:
# Number of  Partners
hts['NumberOfPartners'] = pd.to_numeric(hts['NumberOfPartners'], errors='coerce')

conditions = [
    hts['Age'] <= 10,
    (hts['NumberOfPartners'] >= 2) | (hts['MultiplePartners'] == 'YES'),
    hts['NumberOfPartners'] == 1
]

choices = [
    'NR',
    'MULTIPLE',
    'SINGLE'
]
hts['NumberOfPartners'] = np.select(conditions, choices, default='nan')

#display(hts['NumberOfPartners'].value_counts(dropna=False))

In [42]:
# Filter and count rows where MultiplePartners is 'YES' and NumberOfPartners is not 'MULTIPLE'
nrow_multiple_not_multiple_categorized = hts[
    (hts['MultiplePartners'] == 'YES') &
    (hts['NumberOfPartners'] != 'MULTIPLE')
].shape[0]

print(f"Number of rows where MultiplePartners is 'YES' and NumberOfPartners is not 'MULTIPLE': {nrow_multiple_not_multiple_categorized}")

Number of rows where MultiplePartners is 'YES' and NumberOfPartners is not 'MULTIPLE': 15


In [43]:
#Correct number or rows where multiplepartners is Yes and number of partners is NO
specific_row_index = hts[
    (hts['MultiplePartners'] == 'YES') &
    (hts['NumberOfPartners'] != 'MULTIPLE') #1 Corrected
].index

if not specific_row_index.empty:
    # Set 'MultiplePartners' to 'NR' for the inconsistent row(s)
    hts.loc[specific_row_index, 'MultiplePartners'] = 'NR'
    print(f"Corrected {len(specific_row_index)} row(s): MultiplePartners set to 'NR' where it was 'YES' and NumberOfPartners was not 'MULTIPLE' due to age.\n")

#display(hts.loc[specific_row_index])

# Check and correct Number of rows where NumberOfPartners is 'MULTIPLE' and EverHadSex is 'NO'
inconsistent_partners_sex = hts[
    (hts['NumberOfPartners'] == 'MULTIPLE') &
    (hts['EverHadSex'] == 'NO')
]

num_inconsistent_partners_sex = inconsistent_partners_sex.shape[0]
print(f"Number of rows where NumberOfPartners is 'MULTIPLE' and EverHadSex is 'NO': {num_inconsistent_partners_sex}") #4 cases found and corrected to 'Ýes' under Ever Had Sex

 # Fix the inconsistency: Set EverHadSex to 'YES' for these rows
hts.loc[(hts['NumberOfPartners'] == 'MULTIPLE') & (hts['EverHadSex'] == 'NO'), 'EverHadSex'] = 'YES'
print(f"\nCorrected {num_inconsistent_partners_sex} rows: EverHadSex set to 'YES' where NumberOfPartners was 'MULTIPLE' and EverHadSex was 'NO'.")

Corrected 15 row(s): MultiplePartners set to 'NR' where it was 'YES' and NumberOfPartners was not 'MULTIPLE' due to age.

Number of rows where NumberOfPartners is 'MULTIPLE' and EverHadSex is 'NO': 0

Corrected 0 rows: EverHadSex set to 'YES' where NumberOfPartners was 'MULTIPLE' and EverHadSex was 'NO'.


In [44]:
# Alcohol Sex
hts['AlcoholSex'] = hts['AlcoholSex'].astype(str).str.upper()

conditions = [
    hts['Age'] <= 10,
    hts['AlcoholSex'].str.contains('ALWAYS', na=False),
    hts['AlcoholSex'].str.contains('SOMETIMES', na=False),
    hts['AlcoholSex'].str.contains('NEVER|NOT', na=False)
]

choices = [
    'NR',
    'ALWAYS',
    'SOMETIMES',
    'NEVER'
]

hts['AlcoholSex'] = np.select(conditions, choices, default='nan')

#display(hts['AlcoholSex'].value_counts(dropna=False))

In [45]:
#Money Sex
hts['MoneySex'] = hts['MoneySex'].astype(str).str.upper()

conditions = [
    hts['Age'] <= 10,
    hts['MoneySex'].str.contains('YES', na=False),
    hts['MoneySex'].str.contains('NO', na=False)
]

choices = [
    'NR',
    'YES',
    'NO'
]

hts['MoneySex'] = np.select(conditions, choices, default='nan')

#display(hts['MoneySex'].value_counts(dropna=False))



In [46]:
# Calculate the number of rows where MoneySex is 'YES' and EverHadSex is 'NO'
num_money_sex_no_sex = hts[
    (hts['MoneySex'] == 'YES') &
    (hts['EverHadSex'] == 'NO')
].shape[0]

print(f"Number of rows where MoneySex is 'YES' and EverHadSex is 'NO': {num_money_sex_no_sex}")

Number of rows where MoneySex is 'YES' and EverHadSex is 'NO': 0


In [47]:
#Condom Burst
hts['CondomBurst'] = hts['CondomBurst'].astype(str).str.upper()

conditions = [
    hts['Age'] <= 10,
    hts['CondomBurst'].str.contains('YES', na=False),
    hts['CondomBurst'].str.contains('NO', na=False)
]

choices = [
    'NR',
    'YES',
    'NO'
]

hts['CondomBurst'] = np.select(conditions, choices, default='nan')
#display(hts['CondomBurst'].value_counts(dropna=False))

In [48]:
# Calculate the number of rows where CondomBurst is 'YES' and EverHadSex is 'NO'
num_condom_burst_no_sex = hts[
    (hts['CondomBurst'] == 'YES') &
    (hts['EverHadSex'] == 'NO')
].shape[0]

print(f"Number of rows where CondomBurst is 'YES' and EverHadSex is 'NO': {num_condom_burst_no_sex}")

Number of rows where CondomBurst is 'YES' and EverHadSex is 'NO': 0


In [49]:
#Quick check on the shape of dataset
rows,cols = hts.shape
print(f"There are {rows} rows and {cols} columns in the hts table")

There are 2339903 rows and 126 columns in the hts table


In [50]:
# Unprotected Sex with partner with unknown HIV status
hts['UnknownStatusPartner'] = hts['UnknownStatusPartner'].astype(str).str.upper()

conditions = [
    hts['Age'] <= 10,
    hts['UnknownStatusPartner'].str.contains('YES', na=False),
    hts['UnknownStatusPartner'].str.contains('NO', na=False)
]

choices = [
    'NR',
    'YES',
    'NO'
]
hts['UnknownStatusPartner'] = np.select(conditions, choices, default='nan')
#display(hts['UnknownStatusPartner'].value_counts(dropna=False))


In [51]:
# Calculate the number of rows where UnknownStatusPartner is 'YES' and EverHadSex is 'NO'
num_USP_no_sex= hts[
    (hts['UnknownStatusPartner'] == 'YES') &
    (hts['EverHadSex'] == 'NO')
].shape[0]

print(f"Number of rows where UnknownStatusPartner is 'YES' and EverHadSex is 'NO': {num_USP_no_sex}")

Number of rows where UnknownStatusPartner is 'YES' and EverHadSex is 'NO': 0


In [52]:
# Unprotected Sex with partner with known HIV status
hts['KnownStatusPartner'] = hts['KnownStatusPartner'].astype(str).str.upper

condtions = [
    hts['Age'] <= 10,
    hts['KnownStatusPartner'].str.contains('YES', na=False),
    hts['KnownStatusPartner'].str.contains('NO', na=False)
]

choices = [
    'NR',
    'YES',
    'NO'
]

hts['KnownStatusPartner'] = np.select(conditions, choices, default='nan')
#display(hts['KnownStatusPartner'].value_counts(dropna=False))

In [53]:
# Calculate the number of rows where knownStatusPartner is 'YES' and EverHadSex is 'NO'
num_UP_no_sex= hts[
    (hts['KnownStatusPartner'] == 'YES') &
    (hts['EverHadSex'] == 'NO')
].shape[0]

print(f"Number of rows where knownStatusPartner is 'YES' and EverHadSex is 'NO': {num_UP_no_sex}")

Number of rows where knownStatusPartner is 'YES' and EverHadSex is 'NO': 0


In [54]:
# Pregnant
hts['Pregnant'] = hts['Pregnant'].astype(str).str.upper()

conditions = [
    hts['Age'] <= 10,
    hts['Sex'].str.upper() == 'MALE',  # Assuming 'Sex' column stores gender, and 'MALE' for male
    hts['Age'] > 50,
    hts['Pregnant'].str.contains('YES', na=False),
    hts['Pregnant'].str.contains('NO', na=False)
]

choices = [
    'NR',
    'NR',
    'NR',
    'YES',
    'NO'
]

hts['Pregnant'] = np.select(conditions, choices, default='nan')

#display(hts['Pregnant'].value_counts(dropna=False))

In [55]:
#BreastfeedingMother
hts['BreastfeedingMother'] = hts['BreastfeedingMother'].astype(str).str.upper()
conditions = [
    hts['Age'] <=10,
    hts['Sex'].str.upper() == 'MALE',
    hts['Age'] > 50,
    hts['BreastfeedingMother'].str.contains('YES', na=False),
    hts['BreastfeedingMother'].str.contains('NO', na=False),
    hts['BreastfeedingMother'].str.contains('DECLINED', na=False)
]

choices = [
    'NR',
    'NR',
    'NR',
    'YES',
    'NO',
    'DECLINED'
]
hts['BreastfeedingMother'] = np.select(conditions, choices, default='nan')
#display(hts['BreastfeedingMother'].value_counts(dropna=False))

In [56]:
# Recently Experienced GBV
hts['ExperiencedViolenceScreening'] = hts['ExperiencedViolenceScreening'].astype(str).str.upper()


conditions_gbv = [
    hts['Age'] <= 10, # Not relevant for children
    hts['ExperiencedViolenceScreening'].str.contains('YES', na=False),
    hts['ExperiencedViolenceScreening'].str.contains('NO', na=False)
]

choices_gbv = [
    'NR',
    'YES',
    'NO'
]

hts['ExperiencedViolenceScreening'] = np.select(conditions_gbv, choices_gbv, default='nan')

# Create cross-tabulation tables (proportions)

prop_table_gbv_sex = pd.crosstab(hts['Sex'], hts['ExperiencedViolenceScreening'], normalize='index')
prop_table_age_gbv = pd.crosstab(hts['Age'], hts['ExperiencedViolenceScreening'], normalize='index')
prop_table_gbv_poptype = pd.crosstab(hts['PopulationType'], hts['ExperiencedViolenceScreening'], normalize='index')

#display(hts['ExperiencedViolenceScreening'].value_counts(dropna=False))



In [57]:
#Currently on PrEP
hts['CurrentlyOnPrep'] = hts['CurrentlyOnPrep'].astype(str).str.upper()
conditions = [
    hts['Age'] <= 10,
    hts['CurrentlyOnPrep'].str.contains('YES', na=False),
    hts['CurrentlyOnPrep'].str.contains('NO', na=False)
]
choices = [
    'NR',
    'YES',
    'NO'
]
hts['CurrentlyOnPrep'] = np.select(conditions, choices, default='nan')
#display(hts['CurrentlyOnPrEP'].value_counts(dropna=False))
prep_by_sex = hts.groupby('Sex')['CurrentlyOnPrep'].value_counts(normalize=True).unstack(fill_value=0)


In [58]:
# Currently has STI
hts['CurrentlyHasSTI'] = hts['CurrentlyHasSTI'].astype(str).str.upper()

conditions = [
    hts['Age'] <= 10,
    hts['CurrentlyHasSTI'].str.contains('YES', na=False),
    hts['CurrentlyHasSTI'].str.contains('NO', na=False)
]

choices = [
    'NR',
    'YES',
    'NO'
]

hts['CurrentlyHasSTI'] = np.select(conditions, choices, default='nan')

#display(hts['CurrentlyHasSTI'].value_counts(dropna=False))

In [59]:
# Create cross-tabulation tables for CurrentlyHasSTI
cross_tab_sti_sex = pd.crosstab(hts['Sex'], hts['CurrentlyHasSTI'], normalize='index')
cross_tab_sti_usp = pd.crosstab(hts['UnknownStatusPartner'], hts['CurrentlyHasSTI'], normalize='index')


In [ ]:
# Ever Shared Needle while injecting drugs with someone else?
hts['SharedNeedle'] = hts['SharedNeedle'].astype(str).str.upper()

conditions = [
    hts['Age'] <= 10,
    hts['KeyPopulation'] != 'PWID',
    hts['SharedNeedle'].str.contains('YES', na=False),
    hts['SharedNeedle'].str.contains('NO', na=False)
]

choices = [
    'NR',
    'NR',
    'YES',
    'NO'
]

hts['SharedNeedle'] = np.select(conditions, choices, default='nan')
#display(hts['SharedNeedle'].value_counts(dropna=False))

In [ ]:

# Only asked for PWID?
#Number of individuals who shared needles and are PWID
num_shared_needle_pwid = hts[(hts['SharedNeedle'] == "YES") & (hts['KeyPopulation'] == "PWID")].shape[0]
print(f"Number of individuals who shared needles and are PWID: {num_shared_needle_pwid}")

#Number of individuals who are PWID
num_shared=hts[hts['KeyPopulation'] == 'PWID'].shape[0]
print(f"Number of individuals who are PWID: {num_shared}")

In [ ]:
# NeedleStickInjuries
hts['NeedleStickInjuries'] = hts['NeedleStickInjuries'].astype(str).str.upper()

conditions = [
    hts['Age'] <= 10,
    hts['NeedleStickInjuries'].str.contains('YES', na=False),
    hts['NeedleStickInjuries'].str.contains('NO', na=False)
]

choices = [
    'NR',
    'YES',
    'NO'
]

hts['NeedleStickInjuries'] = np.select(conditions, choices, default='nan')

# display(hts['NeedleStickInjuries'].value_counts(dropna=False))

In [ ]:
# Only asked for PWID?
#Number of individuals with needlesstickinjuries and are PWID
num_needleinjury_pwid = hts[(hts['NeedleStickInjuries'] == "YES") & (hts['KeyPopulation'] == "PWID")].shape[0]
print(f"Number of individuals with needlesstickinjuries and are PWID: {num_needleinjury_pwid}")

In [ ]:
#Traditional Procedures
hts['TraditionalProcedures'] = hts['TraditionalProcedures'].astype(str).str.upper()

conditions = [
    hts['Age'] <= 10,
    hts['TraditionalProcedures'].str.contains('YES', na=False),
    hts['TraditionalProcedures'].str.contains('NO', na=False)
]

choices = [
    'NR',
    'YES',
    'NO'
]

hts['TraditionalProcedures'] = np.select(conditions, choices, default='nan')
# display(hts['TraditionalProcedures'].value_counts(dropna=False))

In [ ]:
#Mother's HIV Status - asked of kids only
hts['MothersStatus'] = hts['MothersStatus'].astype(str).str.upper()

conditions = [
    hts['Age'] >= 15,
    hts['MothersStatus'].str.contains('POSITIVE', na=False),
    hts['MothersStatus'].str.contains('NEGATIVE', na=False),
    hts['MothersStatus'].str.contains('UNKNOWN', na=False)
]

choices = [
    'NR',
    'POSITIVE',
    'NEGATIVE',
    'UNKNOWN'
]

hts['MothersStatus'] = np.select(conditions, choices, default='nan')
# display(hts['MothersStatus'].value_counts(dropna=False))

In [ ]:
# Referred for testing
hts['ReferredForTesting'] = hts['ReferredForTesting'].astype(str).str.upper()

conditions = [
    hts['ReferredForTesting'].str.contains('YES', na=False),
    hts['ReferredForTesting'].str.contains('NO', na=False)
]

choices = [
    'YES',
    'NO'
]

hts['ReferredForTesting'] = np.select(conditions, choices, default='nan')
# display(hts['ReferredForTesting'].value_counts(dropna=False))

In [ ]:
#display(hts['ReferredForTesting'].value_counts(dropna=False))
hts['ReferredForTesting']= hts['ReferredForTesting'].astype(str).str.upper()
hts['ReceivedServices'].value_counts(dropna=False)

In [ ]:
# Received Services - prep, pep, tb, sti
display(hts['ReceivedServices'])

In [ ]:
# Four values, create binaries for each
hts['ReceivedServices'] = hts['ReceivedServices'].astype(str).str.upper()


def create_service_flag(df, service_col, keyword):
    conditions = [
        df[service_col].str.contains(keyword, na=False),
        ~df[service_col].isin(['NULL', '', 'NAN'])
    ]
    choices = ['YES', 'NO']
    return np.select(conditions, choices, default='NR')

# Create new binary columns for each service
hts['ReceivedPrEP'] = create_service_flag(hts, 'ReceivedServices', 'PREP') # 32K
hts['ReceivedPEP'] = create_service_flag(hts, 'ReceivedServices', 'PEP') # 30K
hts['ReceivedTB'] = create_service_flag(hts, 'ReceivedServices', 'TB') #33K
hts['ReceivedSTI'] = create_service_flag(hts, 'ReceivedServices', 'STI') #30K


In [ ]:
print("ReceivedPrEP Distribution:")
display(hts['ReceivedPrEP'].value_counts(dropna=False))

print("\nReceivedPEP Distribution:")
display(hts['ReceivedPEP'].value_counts(dropna=False))

print("\nReceivedTB Distribution:")
display(hts['ReceivedTB'].value_counts(dropna=False))

print("\nReceivedSTI Distribution:")
display(hts['ReceivedSTI'].value_counts(dropna=False))

In [ ]:
#Quick Check
# no one currently on PrEP said they haven't received prep services
crosstab_prep_status = pd.crosstab(hts['ReceivedPrEP'], hts['CurrentlyOnPrep'], dropna=False)

# no one currently on PEP said they haven't received pep services
crosstab_pep_status = pd.crosstab(hts['ReceivedPEP'], hts['CurrentlyOnPep'], dropna=False)

# 2 patients currently on STI said they haven't received STI services
crosstab_STI_status = pd.crosstab(hts['ReceivedSTI'], hts['CurrentlyHasSTI'], dropna=False)

# 24214 patients with presumed TB said they haven't received TB services
crosstab_TB_status = pd.crosstab(hts['ReceivedTB'], hts['TBStatus'], dropna=False)

In [ ]:
# Type GBV sexual, emotional and physical
#display(hts['ViolenceScreeningType'].value_counts(dropna=False))

In [ ]:
# Three values, create binaries for each


hts['ViolenceScreeningType'] = hts['ViolenceScreeningType'].astype(str).str.upper()


conditions_sexual = [
    hts['ViolenceScreeningType'].str.contains('SEXUAL', na=False),
    hts['ExperiencedViolenceScreening'] == 'NO',
    ~hts['ViolenceScreeningType'].isin(['NULL', '', 'NAN'])
]
choices_sexual = ['YES', 'NO', 'NO']
hts['GBVSexual'] = np.select(conditions_sexual, choices_sexual, default='NR')


conditions_physical = [
    hts['ViolenceScreeningType'].str.contains('PHYSICAL', na=False),
    hts['ExperiencedViolenceScreening'] == 'NO',
    ~hts['ViolenceScreeningType'].isin(['NULL', '', 'NAN'])
]
choices_physical = ['YES', 'NO', 'NO']
hts['GBVPhysical'] = np.select(conditions_physical, choices_physical, default='NR')


conditions_emotional = [
    hts['ViolenceScreeningType'].str.contains('EMOTIONAL', na=False),
    hts['ExperiencedViolenceScreening'] == 'NO',
    ~hts['ViolenceScreeningType'].isin(['NULL', '', 'NAN'])
]
choices_emotional = ['YES', 'NO', 'NO']
hts['GBVEmotional'] = np.select(conditions_emotional, choices_emotional, default='NR')


In [ ]:
crosstab_Emotional_status = pd.crosstab(hts['ExperiencedViolenceScreening'], hts['GBVEmotional'], dropna=False) # 8127 experienced VS but not GBVEmotional
crosstab_Physical_status = pd.crosstab(hts['ExperiencedViolenceScreening'], hts['GBVPhysical'], dropna=False) #16958 experienced VS but not GBVPhysical
crosstab_Sexual_status = pd.crosstab(hts['ExperiencedViolenceScreening'], hts['GBVSexual'], dropna=False) #2426 experienced VS but not GBVSexual

#Number of individuals who did not experience GBV (or status is NA) and  GBV types are 'YES': 0
#Number of individuals who experienced GBV but did not report any specific GBV type: 37504

In [ ]:
# Count rows where ExperiencedViolenceScreening is NA or 'NO', and no specific GBV type is 'YES'
num_rows_no_gbv_flags = hts[
    (hts['ExperiencedViolenceScreening'].isna() | (hts['ExperiencedViolenceScreening'] == 'NO')) &
    (hts['GBVEmotional'] == 'YES') &
    (hts['GBVPhysical'] == 'YES') &
    (hts['GBVSexual'] == 'YES')
].shape[0]

print(f"Number of individuals who did not experience GBV (or status is NA) and  GBV types are 'YES': {num_rows_no_gbv_flags}")

In [ ]:
#Number of individuals who experienced GBV but did not report any specific GBV type
num_inconsistent_gbv_screening = hts[
    (hts['ExperiencedViolenceScreening'] == 'YES') &
    (hts['GBVEmotional'] != 'YES') &
    (hts['GBVPhysical'] != 'YES') &
    (hts['GBVSexual'] != 'YES')
].shape[0]

print(f"Number of individuals who experienced GBV but did not report any specific GBV type: {num_inconsistent_gbv_screening}")

In [ ]:
#gender
#display(hts['Sex'].value_counts(dropna=False))

hts['Sex'] = hts['Sex'].astype(str).str.upper()
hts['Sex'] = hts['Sex'].str.replace('FEMALE', 'F')
hts['Sex'] = hts['Sex'].str.replace('MALE', 'M')

#display(hts['Sex'].value_counts(dropna=False))


In [ ]:
# MaritalStatus - Married, Polygamous, Separated/Divorced, Single, Minor
#display(hts['MaritalStatus'].value_counts(dropna=False))

hts['MaritalStatus'] = hts['MaritalStatus'].astype(str).str.upper()

conditions = [
    hts['Age'] < 15,
    hts['MaritalStatus'].str.contains('DIVORCED|SEPARATED', na=False),
    hts['MaritalStatus'].str.contains('POLYGAMOUS', na=False),
    (hts['MaritalStatus'].str.contains('NOT|NEVER|SINGLE|NO', na=False)) & (hts['Age'] >= 15),
    hts['MaritalStatus'].str.contains('MARRIED|LIVING|YES|COHABITING', na=False)
]

choices = [
    "MINOR",
    "DIVORCED",
    "POLYGAMOUS",
    "SINGLE",
    "MARRIED"
]

hts['MaritalStatus'] = np.select(conditions, choices, default=hts['MaritalStatus'])

#display(hts['MaritalStatus'].value_counts(dropna=False))

In [ ]:
# PatientDisabled
hts['PatientDisabled'] = hts['PatientDisabled'].astype(str).str.upper()

conditions = [
    hts['PatientDisabled'].str.contains('NO', na=False),
    hts['PatientDisabled'].str.contains('YES', na=False)
]

choices = [
    "YES",
    "NO"
]

hts['PatientDisabled'] = np.select(conditions, choices, default=hts['PatientDisabled'])
#display(hts['PatientDisabled'].value_counts(dropna=False))

cross_tab_patient_disabled_population_type = pd.crosstab(hts['PatientDisabled'], hts['PopulationType']) #64376 are GP, 43141 are KP and 9817 are Priority

In [ ]:
# Ever Tested for HIV

hts['EverTestedForHiv'] = hts['EverTestedForHiv'].astype(str).str.upper()

conditions = [
    hts['EverTestedForHiv'].str.contains('YES', na=False),
    hts['EverTestedForHiv'].str.contains('NO', na=False)
]

choices = [
    "YES",
    "NO"
]

hts['EverTestedForHiv'] = np.select(conditions, choices, default=hts['EverTestedForHiv'])

In [ ]:
# Months since last test
hts['MonthsSinceLastTest'] = pd.to_numeric(hts['MonthsSinceLastTest'], errors='coerce')

conditions = [
    hts['EverTestedForHiv'] != 'YES',
    (hts['MonthsSinceLastTest'] >= 0) & (hts['MonthsSinceLastTest'] <= 6),
    (hts['MonthsSinceLastTest'] >= 7) & (hts['MonthsSinceLastTest'] <= 12),
    (hts['MonthsSinceLastTest'] >= 13) & (hts['MonthsSinceLastTest'] <= 24),
    hts['MonthsSinceLastTest'] > 24
]

choices = [
    'NR',
    'LASTSIXMONTHS',
    'SEVENTOTWELVE',
    'ONETOTWOYEARS',
    'MORETHANTWOYEARS'
]

hts['MonthsSinceLastTest'] = np.select(conditions, choices, default='nan')


#display(hts['MonthsSinceLastTest'].value_counts(dropna=False))

In [ ]:
# Client Tested As
#display(hts['ClientTestedAs'].value_counts(dropna=False))

hts['ClientTestedAs'] =hts['ClientTestedAs'].astype(str).str.upper()

consitions = [
    hts['ClientTestedAs'].str.contains('COUPLE', na=False),
    hts['ClientTestedAs'].str.contains('INDIVIDUAL', na=False),
]
choices= [
    'COUPLE',
    'INDIVIDUAL'
]
hts['ClientTestedAs'] = np.select(consitions, choices, default='nan')



In [ ]:
hts['TbScreening'].value_counts()

In [ ]:
#TB Screening
hts['TbScreening'] = hts['TbScreening'].astype(str).str.upper()
hts['TbScreening'] = hts['TbScreening'].str.replace(' ', '')

conditions = [
    hts['TbScreening'].str.contains('PRESUMED', na=False),
    hts['TbScreening'].str.contains('CONFIRMED', na=False),
   hts['TbScreening'].str.contains('NOTBSIGNS', na=False)

]

choices = [
    'PRESUMEDTB',
    'CONFIRMEDTB',
    'NOPRESUMEDTB',

]

hts['TbScreening'] = np.select(conditions, choices, default='nan') # Changed default to 'nan' string
#display(hts['Tbscreening'].value_counts(dropna=False))

In [ ]:
#display(hts['Tbscreening'], hts['ReceivedTB'].value_counts(dropna=False))

#cross_tab_Tb = pd.crosstab(hts['Tbscreening'], hts['ReceivedTB'])
#display(cross_tab_Tb)

In [ ]:
# Client Self Tested
hts['ClientSelfTested'] = hts['ClientSelfTested'].astype(str).str.upper()

conditions = [
    hts['ClientSelfTested'].str.contains('YES', na=False),
    hts['ClientSelfTested'].str.contains('NO', na=False)
]

choices = [
    'YES',
    'NO'
]

hts['ClientSelfTested'] = np.select(conditions, choices, default='nan')
# display(hts['ClientSelfTested'].value_counts(dropna=False))

In [ ]:
# Couple Discordant
hts['CoupleDiscordant'] = hts['CoupleDiscordant'].astype(str).str.upper()

conditions = [
    hts['MaritalStatus'].str.contains('SINGLE|MINOR', na=False), # If MaritalStatus is SINGLE or MINOR
    hts['CoupleDiscordant'].str.contains('YES', na=False),
    hts['CoupleDiscordant'].str.contains('NO', na=False)
]

choices = [
    'NR',
    'YES',
    'NO'
]

hts['CoupleDiscordant'] = np.select(conditions, choices, default='nan')
#display(hts['CoupleDiscordant'].value_counts(dropna=False))

pd.crosstab(hts['CoupleDiscordant'], hts['ClientTestedAs'], dropna=False) #most discodant couples are tested as individuals

In [ ]:
#Contact with TB Case
hts['ContactWithTBCase'] = hts['ContactWithTBCase'].astype(str).str.upper()

conditions = [
    hts['Age'] <= 10,
    hts['ContactWithTBCase'].str.contains('YES', na=False),
    hts['ContactWithTBCase'].str.contains('NO', na=False)
]

choices = [
    'NR',
    'YES',
    'NO'
]

hts['ContactWithTBCase'] = np.select(conditions, choices, default='nan')
display(hts['ContactWithTBCase'].value_counts(dropna=False))

In [ ]:
#Lethargy
hts['Lethargy'] = hts['Lethargy'].astype(str).str.upper()

conditions = [
    hts['Age'] <= 10,
    hts['Lethargy'].str.contains('YES', na=False),
    hts['Lethargy'].str.contains('NO', na=False)
]

choices = [
    'NR',
    'YES',
    'NO'
]

hts['Lethargy'] = np.select(conditions, choices, default='nan')
display(hts['Lethargy'].value_counts(dropna=False))

In [ ]:
#Entry Point
hts['EntryPoint'] = hts['EntryPoint'].astype(str).str.upper()

conditions = [
    hts['EntryPoint'].isin(['5485', 'IPD', 'IN PATIENT DEPARTMENT']),
    hts['EntryPoint'].isin(['160542', 'OPD']),
    hts['EntryPoint'].isin(['162181', 'PEDIATRIC', 'PEADIATRIC CLINIC']),
    hts['EntryPoint'].isin(['160538', 'PMTCT_ANC', 'PMTCT ANC', 'PMTCT - ANC']),
    hts['EntryPoint'].isin(['160456', '1623', 'PMTCT MAT', 'PMTCT PNC', 'PMTCT - MAT', 'PMTCT - PNC']),
    hts['EntryPoint'].isin(['160541', 'TB', 'TB CLINIC']),
    hts['EntryPoint'].isin(['159940', 'VCT']),
    hts['EntryPoint'].isin(['162223', 'VMMC']),
    hts['EntryPoint'].isin([
        '5622', '160552', '162050', '159938', '159939', '160546', '160522', '163096',
        'MOBILE OUTREACH', 'OTHER', 'COMMUNITY TESTING', 'HBTC', 'EMERGENCY', 'EMPTY', 'STI CLINIC',
        'NUTRITION CLINIC', 'NAN'
    ])
]

choices = [
    "IPD",
    "OPD",
    "PEDIATRIC",
    "PMTCT_ANC",
    "PMTCT_MAT_PNC",
    "TB",
    "VCT",
    "VMMC",
    "OTHER"
]

hts['EntryPoint'] = np.select(conditions, choices, default=hts['EntryPoint'])
display(hts['EntryPoint'].value_counts(dropna=False))

In [ ]:
#hts['EntryPoint']

In [ ]:
#Test Strategy
hts['TestStrategy'] = hts['TestStrategy'].astype(str).str.upper()

conditions = [
    hts['TestStrategy'].isin(['164163', 'HP', 'HP: HOSPITAL PATIENT TESTING', 'HP/PITC']),
    hts['TestStrategy'].isin(['164953', 'NP', 'NP: HTS FOR NON-PATIENTS', 'HTS FOR NON-PATIENTS']),
    hts['TestStrategy'].isin(['164954', 'VI', 'VI:INTEGRATED VCT CENTER', 'VI: INTEGRATED VCT SITES']),
    hts['TestStrategy'].isin(['164955', 'VS', 'VS:STAND ALONE VCT CENTER', 'VS: STAND ALONE VCT CENTER']),
    hts['TestStrategy'].isin(['159938', 'HB', 'HB:HOME BASED TESTING', 'HOME BASED TESTING']),
    hts['TestStrategy'].isin(['159939', 'MO', 'MO: MOBILE OUTREACH HTS', 'MO: MOBILE AND OUTREACH']),
    hts['TestStrategy'].isin(['161557', 'INDEX', 'INDEX TESTING']),
    hts['TestStrategy'].isin(['166606', 'SNS', 'SOCIAL NETWORKS']),
    hts['TestStrategy'].isin(['5622', 'OTHER', 'O:OTHER', 'OTHERS', 'EMPTY', 'NAN'])
]

choices = [
    "HP",
    "NP",
    "VI",
    "VS",
    "HB",
    "MO",
    "INDEX",
    "SNS",
    "OTHER"
]

hts['TestStrategy'] = np.select(conditions, choices, default=hts['TestStrategy'])

display(hts['TestStrategy'].value_counts(dropna=False))

In [ ]:

#Let's day of week (skip month because we don't have a whole year or multiple years)
hts['dayofweek'] = hts['VDate'].dt.dayofweek # Monday=0, Sunday=6

conditions = [
    hts['dayofweek'] == 6, # Sunday
    hts['dayofweek'] == 0, # Monday
    hts['dayofweek'] == 1, # Tuesday
    hts['dayofweek'] == 2, # Wednesday
    hts['dayofweek'] == 3, # Thursday
    hts['dayofweek'] == 4, # Friday
    hts['dayofweek'] == 5  # Saturday
]

choices = [
    "SUNDAY",
    "MONDAY",
    "TUESDAY",
    "WEDNESDAY",
    "THURSDAY",
    "FRIDAY",
    "SATURDAY"
]

hts['dayofweek'] = np.select(conditions, choices, default=hts['dayofweek'])

#display(hts['dayofweek'].value_counts(dropna=False))

In [ ]:
cols_to_drop = ["Dob", "DOB", "present", "cohort", "VDate",
                  "CurrentlyOnPep", "EverHadSex", "MultiplePartners",
                  "DateTestedProvider", "PatientPk", "VisitDate",
                  "PatientDisabled", "ViolenceScreeningType", "ReceivedServices",
                  "RelationshipWithContact", "NumberOfPartners"]

hts = hts.drop(columns=cols_to_drop, errors='ignore')

rows,cols = hts.shape
print(f"There are {rows} rows and {cols} columns in the hts table after dropping columns") #There are 573772 rows and 83 columns in the hts table after dropping columns

                    Let's join with GIS variables

In [ ]:
!pip install pyreadr
import pyreadr

In [ ]:
# Read the rds dataset
result = pyreadr.read_r('gis_features_iit.rds')
gis = result[None]

display(gis.head())

In [ ]:
if 'Latitude' in gis.columns:
    gis = gis.drop(columns=['Latitude'])
if 'Longitude' in gis.columns:
    gis = gis.drop(columns=['Longitude'])

    gis.drop_duplicates(subset=['FacilityCode'], keep='first', inplace=True)

    # Select only numeric columns for mean imputation
    numeric_cols = gis.select_dtypes(include=np.number).columns
    gis[numeric_cols] = gis[numeric_cols].fillna(gis[numeric_cols].mean())

    hts['SiteCode'] = hts['SiteCode'].astype(str)

    hts = pd.merge(hts, gis, left_on="SiteCode", right_on="FacilityCode", how="inner").drop(columns=['SiteCode_y'])

    hts.replace("", np.nan, inplace=True)

rows,cols = hts.shape
print(f"There are {rows} rows and {cols} columns in the hts table after joining with GIS variables")

                                   Missing data imputation

In [ ]:
# We'll create two versions of the dataset
# 1) Keep missing values missing. Some ML models have sophisticated in-built ways of
# dealing with missing values, particularly XGBoost. For others, we'll need to impute.
# 2) Simple imputation - mean and mode. However, we'll only do this for variables that
# are present at least half the time. For very sparse variables, we don't have enough
# of a basis to impute. For these, we'll give missing values a label of MISSING


In [ ]:
# For all imputation, we're going to learn how to impute from the train set only to
# avoid any leakage. So, first step is to split dataset into train-eval-test.
# do 60-20-20 train-val-test split


In [ ]:
import random

random.seed(2231)
np.random.seed(2231)

In [ ]:
from sklearn.model_selection import train_test_split

# Drop rows where 'FinalTestResult' is NaN before splitting for stratification..
hts_cleaned = hts.dropna(subset=['FinalTestResult']).copy()

hts_train, hts_temp_test = train_test_split(hts_cleaned, test_size=0.4, random_state=2231, stratify=hts_cleaned['FinalTestResult'])
hts_val, hts_test = train_test_split(hts_temp_test, test_size=0.5, random_state=2231, stratify=hts_temp_test['FinalTestResult'])

print(f"Original HTS shape: {hts.shape}")
print(f"HTS shape after dropping NaNs in FinalTestResult (for splitting): {hts_cleaned.shape}")
print(f"Training set shape: {hts_train.shape}")
print(f"Validation set shape: {hts_val.shape}")
print(f"Test set shape: {hts_test.shape}")

In [ ]:
# Create sparse versions of the datasets by simply copying the split DataFrames
sparse_train_df = hts_train.copy()
sparse_val_df = hts_val.copy()
sparse_test_df = hts_test.copy()

# Store them in a dictionary for easy access, similar to the R list structure
sparse_datasets = {
    "sparse_train": sparse_train_df,
    "sparse_val": sparse_val_df,
    "sparse_test": sparse_test_df
}

print("Sparse datasets:")
for name, df in sparse_datasets.items():
    print(f"  {name} shape: {df.shape}")

In [ ]:
## Next, simple imputation
# First, identify which variables are present > 50% of the time and should be imputed
# Identify which variables are too sparse and instead will be given a value of MISSING
cols_to_impute = []
cols_to_unknown = []

selected_column_names = list(hts.columns[0:35]) + list(hts.columns[36:48])

for col_name in selected_column_names:
    vals = hts[col_name]

    non_missing_percentage = 100 * (vals.notna().sum() / len(vals))

    #print(col_name)
    #print(round(non_missing_percentage))

    # Condition for cols_to_impute: >50% non-missing AND has some missing values
    if non_missing_percentage > 50 and vals.isna().any():
        cols_to_impute.append(col_name)

    # Condition for cols_to_unknown: >50% missing values
    if (100 - non_missing_percentage) > 50:
        cols_to_unknown.append(col_name)

print(f"\nColumns to impute: {cols_to_impute}")
print(f"Columns to label as unknown: {cols_to_unknown}")

In [ ]:
# This function identifies the mode in one dataframe and imputes to another
# This is critical because we want the mode from the training set only to avoid leakage

def mode_excluding_nr(series):

    filtered_series = series[series != 'NR'].dropna()
    if not filtered_series.empty:
        return filtered_series.mode()[0]
    else:
        return 'UNKNOWN'

def replace_with_mode(df_calc, df_impute, col_name):
    imputation_mode = mode_excluding_nr(df_calc[col_name])

    if pd.api.types.is_numeric_dtype(df_impute[col_name]):
        df_impute[col_name] = df_impute[col_name].fillna(imputation_mode)
    else:
        df_impute[col_name] = df_impute[col_name].fillna(imputation_mode)
    return df_impute

train_simple_py = sparse_datasets['sparse_train'].copy()
val_simple_py = sparse_datasets['sparse_val'].copy()
test_simple_py = sparse_datasets['sparse_test'].copy()

for col_name in cols_to_impute:
    #print(f"  Imputing column: {col_name}")

    train_simple_py = replace_with_mode(train_simple_py, train_simple_py, col_name)
    val_simple_py = replace_with_mode(train_simple_py, val_simple_py, col_name) # Uses mode from train_simple_py
    test_simple_py = replace_with_mode(train_simple_py, test_simple_py, col_name) # Uses mode from train_simple_py

simple = {
    "simple_train": train_simple_py,
    "simple_val": val_simple_py,
    "simple_test": test_simple_py
}

for name, df_imp in simple.items():
    print(f"Missing values in {name} after translation imputation (first 10):\n{df_imp.isnull().sum().head(10)}")


In [ ]:
# Replace nan with Missing
for col_name in cols_to_unknown:
    # Iterate through all three datasets (train, val, test) and replace NaNs with 'MISSING'
    for df_key in simple:
        simple[df_key][col_name] = simple[df_key][col_name].fillna('MISSING')

#display(simple['simple_train'].head())

In [ ]:
# For simple imputations, add binary variables to indicate whether
# value was missing. This is so that we retain information about missingness.

for col_name in cols_to_impute + cols_to_unknown:
    for df_key, original_df in zip(simple.keys(), [hts_train, hts_val, hts_test]):
        simple[df_key][f'{col_name}_IS_MISSING'] = original_df[col_name].isna().astype(int)


In [ ]:
# For simple, add binary variables to indicate whether
#value was missing. This is so that we retain information about missingness.

sparse_cols_from_train = hts_train.columns[hts_train.isnull().any()].tolist()

# Process simple_train
sparse_binary_train = hts_train[sparse_cols_from_train].notna().astype(int)
sparse_binary_train .columns = [f"{col}_bin" for col in sparse_cols_from_train]
sparse_binary_train  = sparse_binary_train .reindex(simple['simple_train'].index)
simple['simple_train'] = pd.concat([simple['simple_train'], sparse_binary_train], axis=1)

# Do the same for val set
sparse_binary_val = hts_val[sparse_cols_from_train].notna().astype(int)
sparse_binary_val.columns = [f"{col}_bin" for col in sparse_cols_from_train]
sparse_binary_val = sparse_binary_val.reindex(simple['simple_val'].index)
simple['simple_val'] = pd.concat([simple['simple_val'], sparse_binary_val], axis=1)

# Repeat for test set
sparse_binary_test = hts_test[sparse_cols_from_train].notna().astype(int)
sparse_binary_test.columns = [f"{col}_bin" for col in sparse_cols_from_train]
sparse_binary_test = sparse_binary_test.reindex(simple['simple_test'].index)
simple['simple_test'] = pd.concat([simple['simple_test'], sparse_binary_test], axis=1)



### Saving Processed DataFrames to CSV

In [ ]:
# download sparse datasets to a CSV file
for name, df in sparse_datasets.items():
    file_name = f"{name}.csv"
    df.to_csv(file_name, index=False)
    print(f"Saved {name} to {file_name}")

print("All sparse datasets saved as CSVs.")

In [ ]:
# Download Sparse datasets to a CSV file
for name, df in simple.items():
    file_name = f"{name}.csv"
    df.to_csv(file_name, index=False)
    print(f"Saved {name} to {file_name}")

print("All simple imputed datasets saved as CSVs.")